In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('Mobile Reviews Sentiment Cleaned.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

Shape: (50000, 21)
Columns: ['review_id', 'customer_name', 'age', 'brand', 'model', 'price_usd', 'currency', 'exchange_rate_to_usd', 'rating', 'sentiment', 'country', 'language', 'review_date', 'verified_purchase', 'battery_life_rating', 'camera_rating', 'performance_rating', 'design_rating', 'display_rating', 'helpful_votes', 'source']


In [2]:
phone_profiles = df.groupby(['brand', 'model']).agg(
    price_usd=('price_usd', 'mean'),
    rating=('rating', 'mean')
).reset_index()

phone_profiles['price_usd'] = phone_profiles['price_usd'].round(2)
phone_profiles['rating'] = phone_profiles['rating'].round(2)

print('Total unique phones:', phone_profiles.shape[0])
print()
print(phone_profiles)

Total unique phones: 22

       brand            model  price_usd  rating
0      Apple        iPhone 13    1105.99    3.15
1      Apple        iPhone 14    1101.99    3.18
2      Apple    iPhone 15 Pro    1100.81    3.08
3      Apple        iPhone SE    1103.50    3.10
4     Google          Pixel 6     808.51    3.11
5     Google         Pixel 7a     807.56    3.12
6     Google          Pixel 8     799.33    3.12
7   Motorola          Edge 50     509.78    3.09
8   Motorola     Moto G Power     504.26    3.12
9   Motorola          Razr 40     506.09    3.10
10   OnePlus      OnePlus 11R     672.79    3.11
11   OnePlus       OnePlus 12     670.14    3.13
12   OnePlus   OnePlus Nord 3     672.88    3.14
13    Realme    Realme 12 Pro     394.36    3.14
14    Realme  Realme Narzo 70     392.29    3.12
15   Samsung       Galaxy A55     906.73    3.09
16   Samsung   Galaxy Note 20     900.94    3.14
17   Samsung       Galaxy S24     895.49    3.07
18   Samsung    Galaxy Z Flip     899.07    

In [3]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(phone_profiles[['price_usd', 'rating']])

print('Scaling done. Shape:', scaled_features.shape)
print('Each row = [scaled_price, scaled_rating] for one phone')

Scaling done. Shape: (22, 2)
Each row = [scaled_price, scaled_rating] for one phone


In [4]:
def recommend_phones(brand, price, rating, top_n=5):

    user_input = pd.DataFrame([[price, rating]], columns=['price_usd', 'rating'])
    user_scaled = scaler.transform(user_input)

    similarity_scores = cosine_similarity(user_scaled, scaled_features)[0]

    results = phone_profiles.copy()
    results['similarity_score'] = similarity_scores.round(4)

    brand_match = results['brand'].str.lower() == brand.lower()

    if not brand_match.any():
        print(f"Brand '{brand}' not found in dataset. Showing recommendations by price and rating only.")

    same_brand = results[brand_match].sort_values('similarity_score', ascending=False)
    other_brands = results[~brand_match].sort_values('similarity_score', ascending=False)

    final_result = pd.concat([same_brand, other_brands]).head(top_n)
    final_result = final_result.reset_index(drop=True)
    final_result.index = final_result.index + 1
    final_result.index.name = 'rank'

    return final_result[['brand', 'model', 'price_usd', 'rating', 'similarity_score']]


print('Recommendation function is ready!')

Recommendation function is ready!


In [5]:
print('=== Test 1: Samsung, $900, Rating 3.1 ===')
print(recommend_phones(brand='Samsung', price=900, rating=3.1))
print()

=== Test 1: Samsung, $900, Rating 3.1 ===
        brand           model  price_usd  rating  similarity_score
rank                                                              
1     Samsung      Galaxy A55     906.73    3.09            0.9772
2     Samsung      Galaxy S24     895.49    3.07            0.8950
3     Samsung   Galaxy Z Flip     899.07    3.13            0.1287
4     Samsung  Galaxy Note 20     900.94    3.14           -0.1365
5       Apple   iPhone 15 Pro    1100.81    3.08            1.0000



In [6]:
print('=== Test 2: OnePlus, $670, Rating 3.2 ===')
print(recommend_phones(brand='OnePlus', price=670, rating=3.2))
print()

=== Test 2: OnePlus, $670, Rating 3.2 ===
        brand           model  price_usd  rating  similarity_score
rank                                                              
1     OnePlus  OnePlus Nord 3     672.88    3.14            0.9850
2     OnePlus      OnePlus 12     670.14    3.13            0.9374
3     OnePlus     OnePlus 11R     672.79    3.11           -0.6744
4       Apple       iPhone 14    1101.99    3.18            0.8208
5     Samsung  Galaxy Note 20     900.94    3.14            0.7631



In [7]:
print('=== Test 3: Apple, $1100, Rating 3.15 ===')
print(recommend_phones(brand='Apple', price=1100, rating=3.15))
print()

=== Test 3: Apple, $1100, Rating 3.15 ===
        brand          model  price_usd  rating  similarity_score
rank                                                             
1       Apple      iPhone 13    1105.99    3.15            1.0000
2       Apple      iPhone 14    1101.99    3.18            0.9540
3       Apple      iPhone SE    1103.50    3.10            0.4152
4       Apple  iPhone 15 Pro    1100.81    3.08            0.0667
5     Samsung  Galaxy Z Flip     899.07    3.13            0.9984



In [8]:
print('=== Test 4: Sony (not in dataset), $500, Rating 3.0 ===')
print(recommend_phones(brand='Sony', price=500, rating=3.0))
print()

=== Test 4: Sony (not in dataset), $500, Rating 3.0 ===
Brand 'Sony' not found in dataset. Showing recommendations by price and rating only.
         brand        model  price_usd  rating  similarity_score
rank                                                            
1     Motorola      Edge 50     509.78    3.09            0.8738
2      Samsung   Galaxy S24     895.49    3.07            0.8560
3      OnePlus  OnePlus 11R     672.79    3.11            0.8493
4       Xiaomi    Mi 13 Pro     449.14    3.09            0.8091
5     Motorola      Razr 40     506.09    3.10            0.7310

